In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
from scipy import sparse

import gseapy as gp
import networkx as nx
import matplotlib.pyplot as plt
import os


# =========================
# 0) 你的数据（已读入可跳过）
# =========================
# group_files = {...}
# adatas = {g: sc.read_h5ad(f) for g, f in group_files.items()}

# =========================
# 1) 你要比较的 pairs（g1 vs g2）
# =========================
pairs = [
    ("pre_nres_HSCLSC",   "post_nres_HSCLSC"),
    ("pre_nres_EMP",      "post_nres_EMP"),
    ("pre_nres_MPPCLP1",  "post_nres_MPPCLP1"),
]

# =========================
# 2) DEG 参数（保留你原设定）
# =========================
USE_LAYER = None
USE_RAW = False
THRESHOLD = 0.0
MIN_POS_CELLS = 20

USE_FDR = False
P_CUTOFF = 0.05
FDR_CUTOFF = 0.05

FC_METHOD = "mean"        # "mean" or "median"
FC_PSEUDOCOUNT = 0.0

OUT_FULL = "positive_only_ALLGENES_full_results.csv"
OUT_SIG  = "positive_only_sig_up_in_g2.csv"

# =========================
# 3) GO / 网络图 参数
# =========================
GO_SETS = ("GO_Biological_Process_2021",)  # 你也可以换 MF/CC
ORGANISM = "Human"
GO_CUTOFF = 0.05
TOP_TERMS = 8

# 你需要的两个版本：FC>1.0 与 FC>1.3
FC_THRESHOLDS_FOR_NETWORK = (1.0, 1.3)

# 输出图文件夹
FIG_DIR = "gene_go_networks"
os.makedirs(FIG_DIR, exist_ok=True)


# =========================
# 4) 工具函数（DEG）
# =========================
def bh_fdr(pvals):
    pvals = np.asarray(pvals, dtype=float)
    qvals = np.full_like(pvals, np.nan, dtype=float)
    ok = np.isfinite(pvals)
    if ok.sum() == 0:
        return qvals
    pv = pvals[ok]
    order = np.argsort(pv)
    ranked = pv[order]
    m = len(ranked)
    q = ranked * m / (np.arange(1, m + 1))
    q = np.minimum.accumulate(q[::-1])[::-1]
    out = np.empty_like(q)
    out[order] = q
    qvals[ok] = out
    return qvals


def get_matrix_and_genes(adata, use_layer=None, use_raw=False):
    if use_raw:
        if adata.raw is None:
            return None, None
        X = adata.raw.X
        genes = np.asarray(adata.raw.var_names)
    elif use_layer is not None:
        if use_layer not in adata.layers:
            return None, None
        X = adata.layers[use_layer]
        genes = np.asarray(adata.var_names)
    else:
        X = adata.X
        genes = np.asarray(adata.var_names)

    if sparse.issparse(X):
        X = X.tocsr()
    else:
        X = np.asarray(X)
    return X, genes


def summary_stat(x, method="mean"):
    if method == "median":
        return float(np.median(x))
    return float(np.mean(x))


def conditional_mw_positive_only_allgenes_okonly(
    ad1, ad2, g1_name, g2_name,
    min_pos_cells=20, threshold=0.0,
    use_layer=None, use_raw=False,
    fc_method="mean",
    fc_pseudocount=0.0
):
    """
    对所有共有基因做 positive-only MWU：
      - 仅使用 >threshold 的细胞
      - 若任一组 positive cells < min_pos_cells：直接丢弃（不输出）
      - FC / log2FC 计算：g2/g1（基于 positive-only）
    """
    X1, genes1 = get_matrix_and_genes(ad1, use_layer=use_layer, use_raw=use_raw)
    X2, genes2 = get_matrix_and_genes(ad2, use_layer=use_layer, use_raw=use_raw)
    if X1 is None or X2 is None:
        return pd.DataFrame()

    common = np.intersect1d(genes1, genes2, assume_unique=False)
    if common.size == 0:
        return pd.DataFrame()

    idx1 = pd.Index(genes1).get_indexer(common)
    idx2 = pd.Index(genes2).get_indexer(common)

    X1c = X1[:, idx1]
    X2c = X2[:, idx2]

    rows = []
    for j, gene in enumerate(common):
        if sparse.issparse(X1c):
            x1_all = X1c[:, j].toarray().ravel()
            x2_all = X2c[:, j].toarray().ravel()
        else:
            x1_all = np.asarray(X1c[:, j]).ravel()
            x2_all = np.asarray(X2c[:, j]).ravel()

        x1_pos = x1_all[x1_all > threshold]
        x2_pos = x2_all[x2_all > threshold]

        if (x1_pos.size < min_pos_cells) or (x2_pos.size < min_pos_cells):
            continue

        u, p = mannwhitneyu(x1_pos, x2_pos, alternative="two-sided")

        stat_g1 = summary_stat(x1_pos, method=fc_method)
        stat_g2 = summary_stat(x2_pos, method=fc_method)

        denom = stat_g1 + fc_pseudocount
        numer = stat_g2 + fc_pseudocount
        fc = (numer / denom) if denom > 0 else np.nan
        log2fc = np.log2(fc) if (np.isfinite(fc) and fc > 0) else np.nan

        rows.append({
            "pair": f"{g1_name} vs {g2_name}",
            "gene": gene,
            "threshold": float(threshold),

            "p_value": float(p),
            "U": float(u),

            f"{g1_name}_n_pos": int(x1_pos.size),
            f"{g2_name}_n_pos": int(x2_pos.size),

            f"{fc_method}_g1_pos": float(stat_g1),
            f"{fc_method}_g2_pos": float(stat_g2),

            "FC_g2_over_g1": float(fc) if np.isfinite(fc) else np.nan,
            "log2FC_g2_over_g1": float(log2fc) if np.isfinite(log2fc) else np.nan,

            "median_pos_diff(g1-g2)": float(np.median(x1_pos) - np.median(x2_pos)),
            "mean_pos_diff(g1-g2)": float(np.mean(x1_pos) - np.mean(x2_pos)),
        })

    return pd.DataFrame(rows)


# =========================
# 5) GO enrichment + 网络绘图
# =========================
def run_go_enrichr(gene_list, gene_sets, organism="Human", cutoff=0.05):
    enr = gp.enrichr(
        gene_list=list(gene_list),
        gene_sets=list(gene_sets),
        organism=organism,
        cutoff=cutoff
    )
    res = enr.results.copy()
    # 兼容列名
    if "Adjusted P-value" not in res.columns:
        for c in res.columns:
            if "Adjusted" in c and "P" in c:
                res["Adjusted P-value"] = res[c]
                break
    if "Genes" not in res.columns:
        for c in res.columns:
            if c.lower() == "genes":
                res["Genes"] = res[c]
                break
    return res


def build_gene_go_network(enrich_df, fc_map, top_terms=8):
    df = enrich_df.sort_values("Adjusted P-value").head(top_terms).copy()

    G = nx.Graph()
    term_nodes, gene_nodes = [], []

    for _, row in df.iterrows():
        term = row["Term"]
        genes_str = row.get("Genes", "")
        genes = [g.strip() for g in str(genes_str).replace(",", ";").split(";") if g.strip()]

        G.add_node(term, node_type="term", hit_genes=len(genes), padj=float(row["Adjusted P-value"]))
        term_nodes.append(term)

        for g in genes:
            if g not in fc_map:
                continue
            if not G.has_node(g):
                G.add_node(g, node_type="gene", fc=float(fc_map[g]))
                gene_nodes.append(g)
            G.add_edge(term, g)

    return G, term_nodes, gene_nodes, df


def plot_gene_go_network(G, term_nodes, gene_nodes, title="", out_png=None, figsize=(12, 9)):
    plt.figure(figsize=figsize)

    pos = nx.spring_layout(G, k=0.7, seed=0)

    term_sizes = [300 + 80 * max(1, G.nodes[t].get("hit_genes", 1)) for t in term_nodes]

    gene_fc = np.array([G.nodes[g].get("fc", 1.0) for g in gene_nodes], dtype=float)
    if len(gene_fc) == 0:
        gene_fc = np.array([1.0])

    vmin = max(1.0, np.percentile(gene_fc, 5))
    vmax = max(vmin + 1e-6, np.percentile(gene_fc, 95))

    nx.draw_networkx_edges(G, pos, alpha=0.35, width=1.2)

    nx.draw_networkx_nodes(
        G, pos,
        nodelist=term_nodes,
        node_size=term_sizes,
        node_color="#D9B26F",
        edgecolors="white",
        linewidths=1.0
    )

    nodes = nx.draw_networkx_nodes(
        G, pos,
        nodelist=gene_nodes,
        node_size=110,
        node_color=[G.nodes[g].get("fc", 1.0) for g in gene_nodes],
        cmap=plt.cm.Blues,
        vmin=vmin,
        vmax=vmax,
        edgecolors="white",
        linewidths=0.6
    )

    # ===== 在 node 外侧放 gene label（径向外移）=====
    label_top_genes = 50  # 你之前已经有这个参数

    # ===== 1) 先画 GO term labels（保证不丢）=====
    term_labels = {t: t for t in term_nodes}
    nx.draw_networkx_labels(
        G, pos,
        labels=term_labels,
        font_size=10
    )

    # ===== 2) 再画 gene labels：只标 top genes，并移动到节点外侧 =====
    if label_top_genes and len(gene_nodes) > 0:
        top_genes = sorted(
            gene_nodes,
            key=lambda g: G.nodes[g].get("fc", 1.0),
            reverse=True
        )[:label_top_genes]

        # 网络中心（用于径向外移）
        cx = np.mean([pos[n][0] for n in G.nodes()])
        cy = np.mean([pos[n][1] for n in G.nodes()])

        offset = 0.05  # 0.03–0.08 自己调

        for g in top_genes:
            x, y = pos[g]
            dx, dy = x - cx, y - cy
            norm = np.hypot(dx, dy)
            if norm == 0:
                continue
            ux, uy = dx / norm, dy / norm

            lx = x + ux * offset
            ly = y + uy * offset

            plt.text(
                lx, ly, g,
                fontsize=10,
                ha="left" if ux > 0 else "right",
                va="center"
            )



    cbar = plt.colorbar(nodes, shrink=0.7)
    cbar.set_label("fold change (g2/g1)", rotation=90)

    plt.title(title, fontsize=16)
    plt.axis("off")
    plt.tight_layout()

    if out_png:
        plt.savefig(out_png, dpi=300)
    plt.show()
    plt.close()

# =========================
# 6) 主流程：DEG + 输出 + Gene→GO 网络
# =========================
all_res = []
for g1, g2 in pairs:
    if g1 not in adatas or g2 not in adatas:
        raise KeyError(f"Missing group in adatas: {g1} or {g2}")

    df_pair = conditional_mw_positive_only_allgenes_okonly(
        adatas[g1], adatas[g2],
        g1, g2,
        min_pos_cells=MIN_POS_CELLS,
        threshold=THRESHOLD,
        use_layer=USE_LAYER,
        use_raw=USE_RAW,
        fc_method=FC_METHOD,
        fc_pseudocount=FC_PSEUDOCOUNT
    )
    all_res.append(df_pair)

res_df = pd.concat(all_res, ignore_index=True)

if res_df.shape[0] == 0:
    print("No genes passed MIN_POS_CELLS filter across all pairs.")
    res_df.to_csv(OUT_FULL, index=False)
    raise SystemExit

# FDR
res_df["FDR_BH"] = bh_fdr(res_df["p_value"].values)

# 显著筛选
if USE_FDR:
    sig_mask = res_df["FDR_BH"].notna() & (res_df["FDR_BH"] < FDR_CUTOFF)
else:
    sig_mask = res_df["p_value"].notna() & (res_df["p_value"] < P_CUTOFF)

# 方向：g2 上调（log2FC>0）
up_in_g2_mask = res_df["log2FC_g2_over_g1"].notna() & (res_df["log2FC_g2_over_g1"] > 0)
sig_up_df = res_df[sig_mask & up_in_g2_mask].copy()

# 保存 DEG
res_df.to_csv(OUT_FULL, index=False)
sig_up_df.to_csv(OUT_SIG, index=False)
print(f"Saved full results: {OUT_FULL}")
print(f"Saved significant up-in-g2: {OUT_SIG}")
print("Rows kept after MIN_POS_CELLS filtering:", res_df.shape[0])
print("Significant & up in g2:", sig_up_df.shape[0])

# =========================
# 7) 对每个 pair 画两套 Gene→GO 网络（FC>1.0 / FC>1.3）
# =========================
for pair_name, dfp in sig_up_df.groupby("pair"):
    # pair_name like "g1 vs g2"
    g1_name, g2_name = pair_name.split(" vs ")
    print(f"\n== {pair_name} ==")

    for fc_th in FC_THRESHOLDS_FOR_NETWORK:
        df_fc = dfp[dfp["FC_g2_over_g1"].notna() & (dfp["FC_g2_over_g1"] > fc_th)].copy()
        if df_fc.shape[0] < 5:
            print(f"  [FC>{fc_th}] genes too few: {df_fc.shape[0]} (skip)")
            continue

        genes = df_fc["gene"].astype(str).tolist()
        fc_map = dict(zip(df_fc["gene"].astype(str), df_fc["FC_g2_over_g1"].astype(float)))

        enr = run_go_enrichr(genes, gene_sets=GO_SETS, organism=ORGANISM, cutoff=GO_CUTOFF)
        if enr is None or enr.empty:
            print(f"  [FC>{fc_th}] enrichment empty (skip)")
            continue

        G, term_nodes, gene_nodes, topdf = build_gene_go_network(enr, fc_map, top_terms=TOP_TERMS)
        if len(term_nodes) == 0 or len(gene_nodes) == 0:
            print(f"  [FC>{fc_th}] network has no nodes (skip)")
            continue

        out_png = os.path.join(
            FIG_DIR,
            f"GeneGO_{g1_name}_vs_{g2_name}_FCgt{str(fc_th).replace('.','p')}.png"
        )

        title = f"{pair_name} | {GO_SETS[0]} | FC(g2/g1) > {fc_th} | genes={len(genes)}"
        plot_gene_go_network(G, term_nodes, gene_nodes, title=title, out_png=out_png)
        selected_go_terms = list(term_nodes)
        for i in selected_go_terms:
            print(i)

        print(f"  saved: {out_png}")